# Contact Validation — Email & Spanish Phone
Validates email format and Spanish phone number format.

- **Email**: standard RFC-like regex
- **Phone**: 9 digits starting with 6/7/8/9, optionally prefixed with +34

**Configure the cell below**, then **Run All**. The final cell adds validation flags — run it manually.

In [ ]:
# ── Configuration ──────────────────────────────────────────────
TABLE_NAME = "{{TABLE_NAME}}"
LAKEHOUSE_NAME = "{{LAKEHOUSE_NAME}}"
# Email columns to validate (set to [] if none)
EMAIL_COLUMNS = {{EMAIL_COLUMNS}}  # e.g., ["email", "correo_electronico"]
# Phone columns to validate (set to [] if none)
PHONE_COLUMNS = {{PHONE_COLUMNS}}  # e.g., ["telefono", "movil"]
OUTPUT_SUFFIX = "_cleaned"
# ───────────────────────────────────────────────────────────────

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import udf
from pyspark.sql.types import *
import re

spark = SparkSession.builder.getOrCreate()
# Read from _cleaned if it exists (previous fixes), else original
try:
    df = spark.table(f"{TABLE_NAME}{OUTPUT_SUFFIX}")
    print(f"Reading from {TABLE_NAME}{OUTPUT_SUFFIX} (previous fixes applied)")
except:
    df = spark.table(TABLE_NAME)
    print(f"Reading from {TABLE_NAME} (original table)")
original_count = df.count()

print(f"Table: {TABLE_NAME}")
print(f"Rows: {original_count:,}")

In [ ]:
# ── Validation UDFs ────────────────────────────────────────────
email_pattern = r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$'
spanish_phone_pattern = r'^(\+34\s?)?[6-9]\d{8}$'

@udf(BooleanType())
def is_valid_email(value):
    if value is None or str(value).strip() == "":
        return False
    return bool(re.match(email_pattern, str(value).strip()))

@udf(BooleanType())
def is_valid_spanish_phone(value):
    if value is None or str(value).strip() == "":
        return False
    cleaned = re.sub(r'[\s\-\.\(\)]', '', str(value).strip())
    return bool(re.match(spanish_phone_pattern, cleaned))

print("Email and phone validation UDFs registered.")

In [ ]:
print("=" * 60)
print("EMAIL VALIDATION")
print("=" * 60)

for col_name in EMAIL_COLUMNS:
    print(f"\n── Email column: {col_name} ──")
    non_null_df = df.where(F.col(col_name).isNotNull() & (F.trim(F.col(col_name)) != ""))
    non_null_count = non_null_df.count()

    if non_null_count == 0:
        print("  All values are null or empty — skipping.")
        continue

    validated = non_null_df.withColumn("_valid_email", is_valid_email(F.col(col_name)))
    invalid_count = validated.where(F.col("_valid_email") == False).count()

    if invalid_count > 0:
        print(f"  ⚠ {invalid_count:,} invalid email addresses found")
        display(validated.where(F.col("_valid_email") == False).select(col_name).distinct().limit(15))
    else:
        print(f"  ✓ All {non_null_count:,} email addresses are valid format")

if not EMAIL_COLUMNS:
    print("\nNo email columns configured — skipping.")

In [ ]:
print("=" * 60)
print("SPANISH PHONE VALIDATION")
print("=" * 60)

for col_name in PHONE_COLUMNS:
    print(f"\n── Phone column: {col_name} ──")
    non_null_df = df.where(F.col(col_name).isNotNull() & (F.trim(F.col(col_name)) != ""))
    non_null_count = non_null_df.count()

    if non_null_count == 0:
        print("  All values are null or empty — skipping.")
        continue

    validated = non_null_df.withColumn("_valid_phone", is_valid_spanish_phone(F.col(col_name)))
    invalid_count = validated.where(F.col("_valid_phone") == False).count()

    if invalid_count > 0:
        print(f"  ⚠ {invalid_count:,} invalid Spanish phone numbers found")
        display(validated.where(F.col("_valid_phone") == False).select(col_name).distinct().limit(15))
    else:
        print(f"  ✓ All {non_null_count:,} phone numbers are valid Spanish format")

if not PHONE_COLUMNS:
    print("\nNo phone columns configured — skipping.")

---
## ⚠ Apply Fix — Add Contact Validation Flags
The cell below adds boolean validation flag columns. Invalid rows are **NOT removed**, only flagged.

**Review the analysis above before running this cell.**

In [ ]:
# ⚠ APPLY FIX — Run manually after review
cleaned_df = df

for col_name in EMAIL_COLUMNS:
    cleaned_df = cleaned_df.withColumn(f"{col_name}_valid_email", is_valid_email(F.col(col_name)))

for col_name in PHONE_COLUMNS:
    cleaned_df = cleaned_df.withColumn(f"{col_name}_valid_phone", is_valid_spanish_phone(F.col(col_name)))

output_table = f"{TABLE_NAME}{OUTPUT_SUFFIX}"
cleaned_df.write.mode("overwrite").format("delta").saveAsTable(
    output_table
)

print(f"Added validation flags for {len(EMAIL_COLUMNS)} email + {len(PHONE_COLUMNS)} phone columns")
print(f"Output: {output_table}")